# Face Shape CNN — Training Notebook (v2, fixed)

This notebook trains a CNN that looks at a **cropped face photo** and guesses its shape:
Heart, Oblong, Oval, Round, or Square.

### Why the last attempt only got ~42% and what is different here

| Problem in the old run | Fix in this notebook |
|---|---|
| Training used the **whole photo** (hair, shoulders, background). The model had to find the face before it could judge the shape. | Every image is **cropped to just the face** first (same idea as the live camera app), then resized to 224x224. |
| Fine-tuning unfroze **54 layers** on only ~4000 images → the model memorised the training set (training loss fell, validation loss stayed flat). | Only the **last 25 layers** are unfrozen, and **BatchNorm layers stay frozen** (this also removes the big accuracy crash you saw at the start of fine-tuning). |
| Weak augmentation (flip + small rotation only). | Adds **zoom, brightness and contrast** changes so the model sees more variety. |
| Training ended on the *last* epoch, not the *best* one. | `EarlyStopping(restore_best_weights=True)` keeps the best epoch. |
| Dropout was low. | Dropout raised to **0.5** in the classification head. |

Run the cells **one by one, top to bottom**. Each code cell has a short plain-English note above it.


## Step 1 — Check the GPU is on

Training on CPU would take hours. Go to **Runtime → Change runtime type → GPU (T4)** before running this.

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus if gpus else "NONE")
if not gpus:
    print("\n>>> WARNING: No GPU. Go to Runtime -> Change runtime type -> GPU, then run this cell again.")

## Step 2 — Mount Google Drive and unzip the dataset

Change `ZIP_PATH` if your zip is not directly in *My Drive*.
The cell then finds the folder that contains `training_set` and `testing_set` automatically (no matter how the zip was packed).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/face_shape_dataset.zip'   # <-- change if needed

import os, zipfile

EXTRACT_TO = '/content/dataset_raw'
if not os.path.exists(EXTRACT_TO):
    print("Unzipping (one-time)...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_TO)
    print("Unzip done.")
else:
    print("Already unzipped, skipping.")

def find_dataset_root(start):
    # Walk down until we find a folder that has BOTH training_set and testing_set inside it
    for root, dirs, files in os.walk(start):
        if 'training_set' in dirs and 'testing_set' in dirs:
            return root
    return None

RAW_DIR = find_dataset_root(EXTRACT_TO)
if RAW_DIR is None:
    raise SystemExit("Could not find training_set/testing_set inside the zip. Check the zip contents.")
print("Dataset root:", RAW_DIR)

CLASSES = ['Heart', 'Oblong', 'Oval', 'Round', 'Square']
for split in ['training_set', 'testing_set']:
    counts = {c: len(os.listdir(os.path.join(RAW_DIR, split, c))) for c in CLASSES}
    print(split, counts)

## Step 3 — Remove corrupted images

A few files in the Kaggle zip are broken JPEGs. This cell opens every image properly and deletes the ones that cannot be read, so training does not crash half-way.

In [ ]:
from PIL import Image

removed = {}
checked = 0
for split in ['training_set', 'testing_set']:
    for c in CLASSES:
        folder = os.path.join(RAW_DIR, split, c)
        for fname in os.listdir(folder):
            path = os.path.join(folder, fname)
            checked += 1
            try:
                with Image.open(path) as im:
                    im.verify()                       # quick header check
                with Image.open(path) as im:
                    im.convert('RGB').load()          # full decode (catches truncated files)
            except Exception:
                os.remove(path)
                removed[(split, c)] = removed.get((split, c), 0) + 1

print(f"Checked {checked} images, removed {sum(removed.values())} corrupted ones.")
for k, v in removed.items():
    print("  removed", v, "from", k)

## Step 4 — Set up a face detector

We use MediaPipe's face detector (the same family your live app uses). If MediaPipe cannot be installed on this Colab runtime, the cell silently falls back to OpenCV's built-in Haar face detector so the notebook still works.

In [ ]:
!pip -q install mediapipe > /dev/null 2>&1 || echo "mediapipe install failed - will use OpenCV fallback"


In [ ]:
import cv2, numpy as np, urllib.request

DETECTOR = None
try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision

    MODEL_PATH = '/content/blaze_face_short_range.tflite'
    if not os.path.exists(MODEL_PATH):
        urllib.request.urlretrieve(
            'https://storage.googleapis.com/mediapipe-models/face_detector/'
            'blaze_face_short_range/float16/1/blaze_face_short_range.tflite',
            MODEL_PATH)
    _opts = vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        min_detection_confidence=0.5)
    _mp_detector = vision.FaceDetector.create_from_options(_opts)
    DETECTOR = 'mediapipe'
except Exception as e:
    print("MediaPipe not usable here -> using OpenCV Haar cascade instead. Reason:", str(e)[:120])
    _haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    DETECTOR = 'haar'

print("Face detector in use:", DETECTOR)


def detect_face_box(img_bgr):
    """Return (x, y, w, h) of the largest face in the image, or None if no face."""
    if DETECTOR == 'mediapipe':
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = _mp_detector.detect(mp_img)
        if not res.detections:
            return None
        best = max(res.detections, key=lambda d: d.bounding_box.width * d.bounding_box.height)
        bb = best.bounding_box
        return (bb.origin_x, bb.origin_y, bb.width, bb.height)
    else:
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        faces = _haar.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
        if len(faces) == 0:
            return None
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        return (int(x), int(y), int(w), int(h))


def crop_face(img_bgr, box, margin=0.25):
    """Square crop around the face with extra margin so forehead/hairline and jaw are included."""
    x, y, w, h = box
    H, W = img_bgr.shape[:2]
    cx, cy = x + w / 2, y + h / 2
    side = max(w, h) * (1 + 2 * margin)
    x1 = int(max(0, cx - side / 2)); y1 = int(max(0, cy - side / 2))
    x2 = int(min(W, cx + side / 2)); y2 = int(min(H, cy + side / 2))
    return img_bgr[y1:y2, x1:x2]

## Step 5 — Crop every image to the face and save a new dataset

This is the most important fix. Every photo becomes a tight 224x224 face crop.
Images where no face is found are skipped (and counted). Takes a few minutes for 5000 images.

In [ ]:
import time

CROP_DIR = '/content/dataset_cropped'
IMG_SIZE = 224
MARGIN = 0.25   # 25% extra around the detected face box

start = time.time()
summary = {}
for split in ['training_set', 'testing_set']:
    for c in CLASSES:
        src = os.path.join(RAW_DIR, split, c)
        dst = os.path.join(CROP_DIR, split, c)
        os.makedirs(dst, exist_ok=True)
        kept = skipped = 0
        for fname in os.listdir(src):
            img = cv2.imread(os.path.join(src, fname))
            if img is None:
                skipped += 1; continue
            box = detect_face_box(img)
            if box is None:
                skipped += 1; continue
            face = crop_face(img, box, MARGIN)
            if face.size == 0:
                skipped += 1; continue
            face = cv2.resize(face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            cv2.imwrite(os.path.join(dst, os.path.splitext(fname)[0] + '.jpg'), face)
            kept += 1
        summary[(split, c)] = (kept, skipped)
        print(f"{split:13s} {c:7s} kept {kept:4d}   skipped (no face) {skipped}")

print(f"\nCropping finished in {(time.time()-start)/60:.1f} minutes.")

Quick visual check — a few random crops per class. They should show mostly face, with a bit of hair and jaw visible.

In [ ]:
import matplotlib.pyplot as plt, random

fig, axes = plt.subplots(len(CLASSES), 5, figsize=(10, 10))
for r, c in enumerate(CLASSES):
    folder = os.path.join(CROP_DIR, 'training_set', c)
    for k, fname in enumerate(random.sample(os.listdir(folder), 5)):
        img = cv2.cvtColor(cv2.imread(os.path.join(folder, fname)), cv2.COLOR_BGR2RGB)
        axes[r, k].imshow(img); axes[r, k].axis('off')
        if k == 0: axes[r, k].set_title(c, loc='left')
plt.tight_layout(); plt.show()

## Step 6 — Load the cropped images into TensorFlow

- `training_set` is split 80/20 into train / validation.
- `testing_set` is kept completely separate and only used for the final score.
- Augmentation (flip, small rotation, zoom, brightness, contrast) is applied **only to training images**, on the fly.

In [ ]:
from tensorflow.keras import layers

BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DIR, 'training_set'), validation_split=0.2, subset='training',
    seed=SEED, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode='categorical')
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DIR, 'training_set'), validation_split=0.2, subset='validation',
    seed=SEED, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode='categorical')
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DIR, 'testing_set'), shuffle=False,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode='categorical')

class_names = train_ds.class_names
print("Classes:", class_names)

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.04),      # about +/- 15 degrees
    layers.RandomZoom(0.12),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

train_ds = (train_ds
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## Step 7 — Build the model (MobileNetV2 + new head)

- `Rescaling(1/127.5, offset=-1)` is exactly MobileNetV2's required preprocessing (pixels 0-255 → -1..1), built into the model so the live app can feed plain RGB pixels.
- The base is called with `training=False` so its BatchNorm layers stay in inference mode — this prevents the accuracy crash seen at the start of fine-tuning last time.
- Dropout 0.5 fights overfitting.

In [ ]:
from tensorflow.keras import models

base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Rescaling(1.0 / 127.5, offset=-1.0)(inputs)   # MobileNetV2 preprocessing
x = base(x, training=False)                              # BatchNorm frozen in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs, name='face_shape_cnn')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary(show_trainable=True)

## Step 8 — Phase 1: train only the new head (base frozen)

Learning rate 0.001, up to 20 epochs. Stops early if validation accuracy has not improved for 5 epochs, and keeps the best weights.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

PHASE1_EPOCHS = 20
es1 = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)

history1 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE1_EPOCHS, callbacks=[es1])

best1 = max(history1.history['val_accuracy'])
print(f"\nPhase 1 best validation accuracy: {best1:.2%}")

## Step 9 — Phase 2: gentle fine-tuning

Unfreeze only the **last 25 layers** of MobileNetV2, keep every BatchNorm layer frozen, and use a very small learning rate (0.00001). Up to 15 epochs with early stopping.

In [ ]:
N_UNFREEZE = 25
base.trainable = True
for layer in base.layers[:-N_UNFREEZE]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for l in base.layers if l.trainable)
print(f"Unfroze {trainable} of {len(base.layers)} base layers (BatchNorm kept frozen).")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

PHASE2_EPOCHS = 15
es2 = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)

history2 = model.fit(train_ds, validation_data=val_ds, epochs=PHASE2_EPOCHS, callbacks=[es2])

best2 = max(history2.history['val_accuracy'])
print(f"\nPhase 2 best validation accuracy: {best2:.2%}   (phase 1 was {best1:.2%})")

## Step 10 — Accuracy and loss curves

Healthy training: the orange (validation) accuracy line should rise along with the blue one, and validation loss should go **down**, not stay flat.

In [ ]:
acc  = history1.history['accuracy'] + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
vloss= history1.history['val_loss'] + history2.history['val_loss']
split_at = len(history1.history['accuracy'])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(acc, label='Training accuracy'); ax[0].plot(vacc, label='Validation accuracy')
ax[0].axvline(split_at, ls='--', c='gray', label='Fine-tuning starts'); ax[0].set_title('Accuracy'); ax[0].legend(); ax[0].set_xlabel('Epoch')
ax[1].plot(loss, label='Training loss'); ax[1].plot(vloss, label='Validation loss')
ax[1].axvline(split_at, ls='--', c='gray', label='Fine-tuning starts'); ax[1].set_title('Loss'); ax[1].legend(); ax[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## Step 11 — Final score on the untouched testing_set

This is the number to put in your report. Screenshot the report and the confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"FINAL TEST ACCURACY: {test_acc:.2%}\n")

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(cm, index=class_names, columns=class_names))

## Step 12 — Save the model and download it

Saves:
- `face_shape_cnn.keras` — the full model (use this in `main.py` with `tf.keras.models.load_model`)
- `class_names.json` — the label order the model outputs
- `face_shape_cnn.tflite` — a smaller, faster copy for CPU-only laptops (optional, useful for the live camera app)

In [ ]:
import json
from google.colab import files

model.save('face_shape_cnn.keras')
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open('face_shape_cnn.tflite', 'wb') as f:
        f.write(tflite_model)
    print("TFLite export OK:", round(len(tflite_model) / 1e6, 1), "MB")
except Exception as e:
    print("TFLite export skipped:", str(e)[:150])

files.download('face_shape_cnn.keras')
files.download('class_names.json')
if os.path.exists('face_shape_cnn.tflite'):
    files.download('face_shape_cnn.tflite')

## Step 13 — Sanity check: reload the saved model and predict a few test images

Proves the saved file works on its own (this is what main.py will do).

In [ ]:
reloaded = tf.keras.models.load_model('face_shape_cnn.keras')

fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, ax in enumerate(axes.flat):
    c = random.choice(class_names)
    folder = os.path.join(CROP_DIR, 'testing_set', c)
    fname = random.choice(os.listdir(folder))
    img = cv2.cvtColor(cv2.imread(os.path.join(folder, fname)), cv2.COLOR_BGR2RGB)
    probs = reloaded.predict(np.expand_dims(img.astype('float32'), 0), verbose=0)[0]
    pred = class_names[int(np.argmax(probs))]
    ax.imshow(img); ax.axis('off')
    ax.set_title(f"true: {c}\npred: {pred} ({probs.max():.0%})",
                 color='green' if pred == c else 'red', fontsize=9)
plt.tight_layout(); plt.show()

## How this plugs into `main.py` (for later)

1. Load once at startup: `model = tf.keras.models.load_model('face_shape_cnn.keras')` and read `class_names.json`.
2. Every N frames (not every frame), take the face bounding box from the MediaPipe landmarks, expand it by the **same 25% margin**, crop, resize to **224x224**.
3. Convert **BGR → RGB** (OpenCV gives BGR; the model was trained on RGB). Keep pixel values as 0-255 floats — the rescaling is already inside the model.
4. `probs = model.predict(face[None, ...])` → `class_names[np.argmax(probs)]`.
5. Run this in a background thread so the camera window never stutters. Keep the old rule-based function as the fallback if the model file is missing.
